# Phase 5 — NB1: Stage 1 Category Detection Training (R5)

**Goal:** Train two Stage 1 experiments and compare:
- **Exp 1 (R5):** Asymmetric Loss (ASL, gamma_neg=4, margin=0.05) — fixes correlated label interference
- **Exp 2 (R5-CatAware):** Category-Aware Attention Pooling — 12 category-specific attention views

**Baseline:** R4 Cat F1 = 0.6844 (global strategy, test set)

**Input:** `lcminhc/semeval-absa-restaurant` (raw XML)

**Output:** Upload `/kaggle/working/outputs_p5_nb1/` as Kaggle dataset `p5-nb1-stage1`
- `stage1_r5_best.pt` (ASL checkpoint)
- `stage1_r5_cataware_best.pt` (Cat-Attention checkpoint)
- `category_detection.jsonl`, `sentiment_records.jsonl`
- Training logs

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml iterative-stratification

In [ ]:
import os
import sys
import json
import shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Wire SemEval XML data
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
assert KAGGLE_INPUT, 'Dataset semeval-absa-restaurant not found'
print(f'SemEval input: {KAGGLE_INPUT}')

os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training', exist_ok=True)
os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant', exist_ok=True)

shutil.copy(f'{KAGGLE_INPUT}/semeval16_restaurants_train.xml',
            'SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training/ABSA16_Restaurants_Train_SB1_v2.xml')
shutil.copy(f'{KAGGLE_INPUT}/semeval16_restaurants_test.xml',
            'SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant/EN_REST_SB1_TEST.xml.gold')
print('XML files in place.')

## 1. Prepare Data

In [ ]:
!python scripts/01_prepare_data.py

In [ ]:
for f in ['category_detection.jsonl', 'sentiment_records.jsonl']:
    path = f'data/processed/{f}'
    if os.path.exists(path):
        with open(path) as fp:
            n = sum(1 for _ in fp)
        print(f'{f}: {n} records')
    else:
        print(f'MISSING: {f}')

In [ ]:
import torch
import gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Exp 1 — Asymmetric Loss (R5)

Config: `stage1_r5.yaml` — ASL (gamma_neg=4, margin=0.05), no pos_weight, multi-label stratified split.

Expected: +2-4pp Cat F1 by reducing gradient interference between correlated labels.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04a_train_stage1.py --config configs/stage1_r5.yaml

In [ ]:
log_path = 'logs/stage1_r5_training.jsonl'
print('=== Exp 1 (ASL) Training Log ===')
if os.path.exists(log_path):
    print(f'{"Epoch":<8} {"Train Loss":<12} {"Cat F1":<10} {"Cat P":<10} {"Cat R":<10}')
    print('-' * 52)
    with open(log_path) as f:
        for line in f:
            rec = json.loads(line)
            print(f"{rec['epoch']:<8} {rec['train_loss']:<12.4f} {rec['category_f1']:<10.4f} "
                  f"{rec['category_precision']:<10.4f} {rec['category_recall']:<10.4f}")
else:
    print('No log found.')

## 3. Exp 2 — Category-Aware Attention Pooling (R5-CatAware)

Config: `stage1_r5_cataware.yaml` — 12 per-category attention queries, encoder_lr=1e-5, head_lr=5e-4.

Expected: +3-7pp Cat F1 by giving each category its own attention view over token sequence.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04a_train_stage1.py --config configs/stage1_r5_cataware.yaml

In [ ]:
log_path = 'logs/stage1_r5_cataware_training.jsonl'
print('=== Exp 2 (Cat-Aware Attention) Training Log ===')
if os.path.exists(log_path):
    print(f'{"Epoch":<8} {"Train Loss":<12} {"Cat F1":<10} {"Cat P":<10} {"Cat R":<10}')
    print('-' * 52)
    with open(log_path) as f:
        for line in f:
            rec = json.loads(line)
            print(f"{rec['epoch']:<8} {rec['train_loss']:<12.4f} {rec['category_f1']:<10.4f} "
                  f"{rec['category_precision']:<10.4f} {rec['category_recall']:<10.4f}")
else:
    print('No log found.')

## 4. Compare Results

In [ ]:
def best_epoch(log_path):
    if not os.path.exists(log_path):
        return None
    best = None
    with open(log_path) as f:
        for line in f:
            rec = json.loads(line)
            if best is None or rec['category_f1'] > best['category_f1']:
                best = rec
    return best

r4_f1 = 0.6844  # R4 baseline (val, per-category threshold tuning)
r5    = best_epoch('logs/stage1_r5_training.jsonl')
r5ca  = best_epoch('logs/stage1_r5_cataware_training.jsonl')

print(f'{"Config":<20} {"Best Epoch":<12} {"Cat F1 (val)":<15} {"vs R4"}')
print('-' * 60)
print(f'{"R4 baseline":<20} {"16":<12} {r4_f1:<15.4f} —')
if r5:
    delta = r5["category_f1"] - r4_f1
    print(f'{"R5 (ASL)":<20} {r5["epoch"]:<12} {r5["category_f1"]:<15.4f} {delta:+.4f}')
if r5ca:
    delta = r5ca["category_f1"] - r4_f1
    print(f'{"R5-CatAware":<20} {r5ca["epoch"]:<12} {r5ca["category_f1"]:<15.4f} {delta:+.4f}')

## 5. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb1'
os.makedirs(output_dir, exist_ok=True)

# Exp 1 checkpoint (ASL)
src = 'checkpoints/stage1_r5/best.pt'
if os.path.exists(src):
    shutil.copy(src, f'{output_dir}/stage1_r5_best.pt')
    print(f'stage1_r5_best.pt: {os.path.getsize(src)/1e6:.1f} MB')

# Exp 2 checkpoint (Cat-Attention)
src = 'checkpoints/stage1_r5_cataware/best.pt'
if os.path.exists(src):
    shutil.copy(src, f'{output_dir}/stage1_r5_cataware_best.pt')
    print(f'stage1_r5_cataware_best.pt: {os.path.getsize(src)/1e6:.1f} MB')

# Data files (needed by NB2 and NB3)
for f in ['category_detection.jsonl', 'sentiment_records.jsonl']:
    src = f'data/processed/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{f}')
        print(f'{f} copied')

# Logs
if os.path.exists('logs'):
    shutil.copytree('logs', f'{output_dir}/logs', dirs_exist_ok=True)
    print('logs/ copied')

print(f'\nOutputs saved to {output_dir}')
print('Upload as Kaggle dataset: p5-nb1-stage1')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_nb1_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_nb1')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_nb1_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')